In [1]:
!pip install -q datasets pytorch-lightning transformers evaluate sacrebleu accelerate


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python3 -m pip install --upgrade pip


# Загрузка датасета и предобработка

In [2]:
import torch
import torch.nn as nn

from typing import List, Optional

from datasets import load_dataset
from tokenizers.processors import TemplateProcessing
from tokenizers import Tokenizer
from tokenizers.models import BPE

from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

from transformers import PreTrainedTokenizerFast, PreTrainedTokenizer

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
books = load_dataset("opus_books", "en-fr")
books["train"] = books["train"].select(range(10000))

books = books.filter(lambda x: len(x['translation']['en']) < 250)
print(books)

Filter: 100%|██████████| 10000/10000 [00:00<00:00, 58299.24 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 8919
    })
})


In [4]:
books['train'][0]

{'id': '0', 'translation': {'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'}}

In [5]:
ALL_SETENCES_FILE = 'all_book_sentences.txt'

with open(ALL_SETENCES_FILE, 'w') as f:
    for item in books['train']:
        f.write(item['translation']['en'] + "\n")
        f.write(item['translation']['fr'] + "\n")

In [6]:
!head all_book_sentences.txt

The Wanderer
Le grand Meaulnes
Alain-Fournier
Alain-Fournier
First Part
PREMIÈRE PARTIE
I
CHAPITRE PREMIER
THE BOARDER
LE PENSIONNAIRE


In [7]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

VOCAB_SIZE = 20000

bpe_trainer = BpeTrainer(special_tokens=["[UNK]", "[BOS]", "[EOS]", "[PAD]"], show_progress=True, vocab_size=VOCAB_SIZE)

tokenizer.pre_tokenizer = Whitespace()

files = [ ALL_SETENCES_FILE ]

tokenizer.train(files, bpe_trainer)

In [8]:
tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[
        ("[BOS]", tokenizer.token_to_id("[BOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]")),
    ],
)

tokenizer.save("tokenizer.json")

In [9]:
tokenizer.encode('[BOS]')

Encoding(num_tokens=3, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [10]:
fast_tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer)
fast_tokenizer.bos_token = "[BOS]"
fast_tokenizer.eos_token = "[EOS]"
fast_tokenizer.pad_token = "[PAD]"
fast_tokenizer.unk_token = "[UNK]"

In [11]:
source_lang = "en"
target_lang = "fr"

def preprocess_function(examples):
    inputs = [example[source_lang] for example in examples["translation"]]
    targets = [example[target_lang] for example in examples["translation"]]
    model_inputs = fast_tokenizer(inputs, text_target=targets, max_length=64, truncation=True, add_special_tokens=True)
    return model_inputs


In [12]:
books_preprocessed = books.map(preprocess_function, batched=True)

Map:   0%|          | 0/8919 [00:00<?, ? examples/s]

Map: 100%|██████████| 8919/8919 [00:02<00:00, 3833.85 examples/s]


In [13]:
books_preprocessed['train'][0]

{'id': '0',
 'translation': {'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'},
 'input_ids': [1, 281, 19049, 1203, 2],
 'token_type_ids': [0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1],
 'labels': [1, 521, 558, 346, 2]}

In [14]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=fast_tokenizer, return_tensors="pt")

In [15]:
fast_tokenizer("test", text_target="test")

{'input_ids': [1, 4207, 2], 'token_type_ids': [0, 0, 0], 'attention_mask': [1, 1, 1], 'labels': [1, 4207, 2]}

In [16]:
fast_tokenizer.eos_token_id

2

In [17]:

data_collator( [ {"input_ids": [ 100, 200, 300 ], "labels": [100,200,300, 400, 500]}, { "input_ids": [ 100, 200, 300,400,500], "labels": [100,200,300,400,500,600]} ] )


{'input_ids': tensor([[100, 200, 300,   3,   3],
        [100, 200, 300, 400, 500]]), 'attention_mask': tensor([[1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1]]), 'labels': tensor([[ 100,  200,  300,  400,  500, -100],
        [ 100,  200,  300,  400,  500,  600]])}

In [ ]:
# ВАЖНО! Эту ячейку в пайплайнах будет копировать в отдельный файл.
# Нельзя изменять названия и конструкторы классов.
# Не стоит изменять сигнатуры методов
# Не стоит добавлять сюда новые ненужные импорты
# Не стоит удалять импорты, которые тут были
# Не стоит добавлять в эту ячейку новые классы

from transformers.modeling_outputs import Seq2SeqModelOutput, Seq2SeqLMOutput

import torch
import torch.nn as nn

from transformers.modeling_outputs import Seq2SeqModelOutput, Seq2SeqLMOutput

from transformers import GenerationConfig, PretrainedConfig, PreTrainedModel
import random
import numpy as np

# Для прохождения тестов не будет требоваться, чтобы модель полностью обучилась
# это может занять много времени. В этой домашке будет достаточно переобучить модель
# на небольшой части датасета.
# 
# Цель домашки -- это реализовать свой трансформер
# Нужно дополнить процесс обучения ниже, заполнить пропуски
# 
# Чтобы домашка была тестируемой, пришлось прибегнуть к таким требованиям
# Кроме того, нет большого практического смысла в обучении такой архитектуры,
# потому что она устарела, лучше обучите трансформер во втором ноутбуке)
# 
# HINT!
# Прописывайте размерности разных тензоров в комментах -- так будет проще разбираться в коде и дебажить
# 
# Черпать вдохновение можно отсюда http://nlp.seas.harvard.edu/2018/04/03/attention.html
# И Attention Is All You Need https://arxiv.org/abs/1706.03762

# PretrainedConfig см тут https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/configuration#transformers.PretrainedConfig
class TransformerAttentionConfig(PretrainedConfig):
    model_type = "rnn"

    r"""
    В классе конфига должны быть описаны все гипер-параметры модели

    Args:
        vocab_size (`int`):
            Размер словаря
        embedding_dim (`int`):
            Размерность эмбэддингов
        hidden_dim (`int`):
            размерность скрытых слоев
        num_layers (`int`):
            количество слоев трансформера (и для енкодера, и для декодера)
        max_length (`int`):
            Максимальная длинна сгенерированной последовательности

    """

    def __init__(
        self,
        vocab_size=20000,
        embedding_dim=128,
        hidden_dim=128,
        num_layers=3,
        max_length=64,
        # Эти токены должны быть предопределены уже в PretrainedConfig
        # https://github.com/huggingface/transformers/blob/cf11493dce0a1d22446efe0d6c4ade02fd928e50/src/transformers/configuration_utils.py#L214
        # pad_token_id=None,
        # bos_token_id=None,
        # eos_token_id=None,
        **kwargs,
    ):
        super().__init__(**kwargs, max_length=max_length)

        assert embedding_dim == hidden_dim, 'this transformer implementation requires embedding_dim to be equals to hidden_dim'

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers


# вот тут про PreTrainedModel  https://github.com/huggingface/transformers/blob/cf11493dce0a1d22446efe0d6c4ade02fd928e50/src/transformers/modeling_utils.py#LL1009C7-L1009C22
class Seq2SeqTransformerAttention(PreTrainedModel):

    config_class = TransformerAttentionConfig
    base_model_prefix = "transformer"
    supports_gradient_checkpointing = False

    def __init__(self, config):

        super().__init__( config )

        self.generation_config = GenerationConfig()
        self.generation_config.max_length = config.max_length

        # начинаем с того, что правильно опишем используемые модули
        # используем self.config.*

        self.embeddings = nn.Embedding( config.vocab_size, config.embedding_dim )
        self.embeddings_dropout = nn.Dropout(p=0.1)

        self.encoder_norm = nn.LayerNorm( config.embedding_dim)

        encoder_layer = nn.TransformerEncoderLayer( d_model=config.embedding_dim, nhead=8 )
        encoder_norm = nn.LayerNorm( config.embedding_dim  )
        self.transformer_encoder = nn.TransformerEncoder( encoder_layer=encoder_layer, num_layers=config.num_layers, norm=encoder_norm )

        decoder_layer = nn.TransformerDecoderLayer( d_model=config.embedding_dim, nhead=8)
        decoder_norm = nn.LayerNorm( config.embedding_dim  )
        self.transformer_decoder = nn.TransformerDecoder( decoder_layer=decoder_layer, num_layers=config.num_layers, norm=decoder_norm )

        self.decoder_labels_linear = nn.Linear( in_features=config.embedding_dim, out_features=config.vocab_size )

        self.criterion = nn.CrossEntropyLoss(ignore_index=self.config.pad_token_id)

        return


    # copy paste from
    # http://nlp.seas.harvard.edu/2018/04/03/attention.html
    def subsequent_mask(self, size, device='cpu'):
        "Mask out subsequent positions."
        attn_shape = (size, size)
        subsequent_mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')
        return (torch.from_numpy(subsequent_mask) != 0).to(device)

    def encode(self, input_embeddings=None, key_padding_mask=None):

        # получаем неконтекстные эмбэддинги
        # получаем контекстные эмбэддинги с помощью self.encoder

        encoder_embeddings = self.transformer_encoder(
            src=input_embeddings,
            src_key_padding_mask=key_padding_mask
        )
        encoder_embeddings = self.encoder_norm(encoder_embeddings)
        return encoder_embeddings

    def decode(self, encoder_outputs=None, lebels_embeddings=None, key_padding_mask=None, encoder_key_padding_mask=None):
        # не забудьте, что нужно обработать сдвиг токенов
        # декодер должен вернуть для каждого текущего токена последующий токен
        # у последнего токена нет последующего тк он последний
        # поэтому последний токен не надо передавать

        # прелесть трансформеров заключается в том, что у них параллелится обучение декодера
        # не надо на каждый токен запускать generate_encoded как это было в RNN
        # это возможно благодаря тому, что мы можем замаскировать будущие токены в механизме внимания
        # декодера с помощью subsequent_mask
        tgt_mask = self.subsequent_mask( lebels_embeddings.shape[1] - 1, device=encoder_outputs.device )

        # не забываем про сдвиг
        tgt = lebels_embeddings[:, :-1, :]
        tgt_key_padding_mask = key_padding_mask[:, :-1] if key_padding_mask is not None else None
        memory_key_padding_mask = encoder_key_padding_mask

        # см доку https://pytorch.org/docs/stable/generated/torch.nn.TransformerDecoder.html
        # осталось правильно передать аргументы
        # не забываем про сдвиг!
        decoder_output = self.transformer_decoder(
            tgt=tgt,
            memory=encoder_outputs,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )

        return decoder_output

    # https://huggingface.co/docs/transformers/v4.28.1/en/main_classes/output#transformers.modeling_outputs.Seq2SeqModelOutput
    def forward(self, input_ids=None, labels=None, attention_mask=None, token_type_ids=None, ) -> Seq2SeqModelOutput:

        # input_ids: [ batch_size, src_seq_len ]
        # labels: [ batch_size, tgt_seq_len ]

        encoder_key_padding_mask = (attention_mask == 0)

        decoder_key_padding_mask = (labels == -100)

        input_embeddings = self.embeddings(input_ids)
        input_embeddings = self.embeddings_dropout(input_embeddings)

        labels[labels == -100] = self.config.pad_token_id
        lebels_embeddings = self.embeddings(labels)
        lebels_embeddings = self.embeddings_dropout(lebels_embeddings) # используется для обучения декодера
        
        # encoder_outputs может еще называться memory

        encoder_outputs = self.encode(
            input_embeddings=input_embeddings,
            key_padding_mask=encoder_key_padding_mask
        )


        decoder_outputs = self.decode(
            encoder_outputs=encoder_outputs,
            lebels_embeddings=lebels_embeddings,
            key_padding_mask=decoder_key_padding_mask,
            encoder_key_padding_mask=encoder_key_padding_mask
        )
        
        # размерность labels_logits: [batch_size, tgt_len-1, vocab_size]
        labels_logits = self.decoder_labels_linear(decoder_outputs)  

        # не забываем про смещение токенов для labels
        loss = self.criterion(
            labels_logits.view(-1, self.config.vocab_size),
            labels[:, 1:].contiguous().view(-1)
        )

        return Seq2SeqLMOutput(
            loss=loss,
            decoder_hidden_states=decoder_outputs,
            encoder_hidden_states=encoder_outputs,
        )

    def generate(self, input_ids=None, attention_mask=None, token_type_ids=None, max_length=None, num_beams=None, **kwargs):
        """
        Метод используется в trainer.evaluate
        """
        
        batch_size = input_ids.shape[0]

        if attention_mask is not None:
            attention_mask = (attention_mask == 0)

        # надо
        # 1. Получить эмбэдддинги
        input_embeddings = self.embeddings(input_ids)
        input_embeddings = self.embeddings_dropout(input_embeddings)

        # 2. Получить Контекстные эмбэддинги (memory) через енкодер
        encoder_outputs = self.encode(
            input_embeddings=input_embeddings,
            key_padding_mask=attention_mask
        )

        # 3. Запустить генерацию generate_encoded
        predicted_tokens_sequences, _ = self.generate_encoded(
            batch_size=batch_size,
            encoder_outputs=encoder_outputs,
            encoder_key_padding_mask=attention_mask,
            max_length=max_length
        )

    
        return predicted_tokens_sequences

    def generate_encoded(self, batch_size=None, encoder_outputs=None, encoder_key_padding_mask=None, max_length=None, num_beams=None, **kwargs):

        generated_tokens = torch.tensor([self.config.bos_token_id] * batch_size, device=encoder_outputs.device).unsqueeze(1) # [ bs, 1 ]

        predicted_tokens_sequences = [ generated_tokens ] # [ bs, 1 ]


        decoder_outputs = [] # [batch_size, current_seq_len, hidden_dim]

        if max_length is None:
            max_length = self.generation_config.max_length

        for token_i in range(max_length):
            # генерим по одному токену
            # greedy decode

            # тут должны быть неконтекстные эмбэддинги для уже сгенерированных токенов
            generated_tokens_embeddings = self.embeddings(generated_tokens)
            generated_tokens_embeddings = self.embeddings_dropout(generated_tokens_embeddings)

            tgt_mask = self.subsequent_mask(
                            generated_tokens_embeddings.size(1),
                            device=encoder_outputs.device
                        )
            decoder_output = self.transformer_decoder(
                tgt=generated_tokens_embeddings,
                memory=encoder_outputs,
                tgt_mask=tgt_mask,
                memory_key_padding_mask=encoder_key_padding_mask
            )

            last_token_outputs = decoder_output[:, -1:, :]
            decoder_outputs.append(last_token_outputs)

            next_token_logits = self.decoder_labels_linear(last_token_outputs.squeeze(1))

            # из логитов надо получить вероятности токенов, как это делали в RNN
            _, next_generated_tokens = torch.max(next_token_logits, dim=-1)
            next_generated_tokens = next_generated_tokens.unsqueeze(1)  

            predicted_tokens_sequences.append(next_generated_tokens)

            # predicted_tokens_sequences расширили, поэтому надо обновить и generated_tokens
            generated_tokens = torch.cat(predicted_tokens_sequences, dim=1)

        # это мы уже делали для RNN, делаем паддинг после EOS токена
        eos_token_indexes = torch.nonzero(generated_tokens == self.config.eos_token_id, as_tuple=False)
        for eos_idx in eos_token_indexes:
            generated_tokens[eos_idx[0], eos_idx[1]:] = self.config.pad_token_id

        return generated_tokens, decoder_outputs

In [23]:
transformer_attention_config = TransformerAttentionConfig(
    pad_token_id=fast_tokenizer.pad_token_id,
    bos_token_id=fast_tokenizer.bos_token_id,
    eos_token_id=fast_tokenizer.eos_token_id,
)
transformer_attention_model = Seq2SeqTransformerAttention(transformer_attention_config)

In [24]:
sum( p.numel() for p in transformer_attention_model.parameters() if p.requires_grad )

6529312

In [25]:
transformer_attention_model.generate( input_ids = torch.arange(15).reshape(3, 5), max_length=64 )

tensor([[    1, 10819, 12965, 16317,  7899,  4898,  3511, 18081,  1399, 14404,
          9444, 11616, 18634, 11567,  2736,  7009, 12074,  3400, 14671,  5788,
         12023,  2736,  7009, 12074,  9384, 14076,   789, 18757, 10609, 18634,
          9080, 18634,  9080, 18634, 18902, 16198, 17772, 10191, 17922, 10566,
          6413, 15093,  9605,  1399, 14404,  6135, 12857,  7958, 11366,  6238,
          5438,  6103,  2113, 16256, 14362,   623, 12023, 18669,  4198,  5682,
          5423, 17704, 18757, 10609, 18634],
        [    1, 18094,  6599, 14450,  1442, 12743, 12285, 14404, 12812,  6016,
         10529,  8597, 16215, 12666, 19780,   533, 14140, 16048,  5100, 17704,
         18757,  4280,   598, 14140,  2106, 15784,  7259,  2130,  5675, 13312,
           598,  9605,  3629,  2130,  2130,  2130,  2130,  2130, 11832, 18419,
           259, 14541,   608, 14239, 14714,  1651, 15091, 10556, 12023, 15170,
         12023,  4183,  3712, 17852, 14489,   259, 14541,  6201,  9605, 12812,
       

In [26]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [27]:
import numpy as np
import evaluate

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = fast_tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, fast_tokenizer.pad_token_id)
    decoded_labels = fast_tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    print('decoded_preds[0]', fast_tokenizer.batch_decode(preds, skip_special_tokens=False)[0])
    print('decoded_labels[0]', decoded_labels[0])

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != fast_tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

In [29]:
transformer_attention_config = TransformerAttentionConfig(
    pad_token_id=fast_tokenizer.pad_token_id,
    bos_token_id=fast_tokenizer.bos_token_id,
    eos_token_id=fast_tokenizer.eos_token_id,
)
transformer_attention_model = Seq2SeqTransformerAttention(transformer_attention_config)


training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_opus_books_model",
    eval_strategy="steps",
    eval_steps=500,
    learning_rate=1e-4,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=500,
    predict_with_generate=True,
    logging_steps=25,
)

trainer = Seq2SeqTrainer(
    model=transformer_attention_model,
    args=training_args,
    train_dataset=books_preprocessed["train"].select(range(128)),
    eval_dataset=books_preprocessed["train"].select(torch.tensor(range(128))),
    tokenizer=fast_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_2440/3174689135.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [30]:
# Скор можно выбить за 5 минут
# Если обучение у вас идет сильно дольше, скорее всего, что-то пошло не так
trainer.train()

Step,Training Loss,Validation Loss,Bleu,Gen Len
500,4.514700,4.301549,0.647800,9.953100
1000,2.904700,2.537061,5.709800,20.531200
1500,1.939400,1.502383,20.230200,22.984400
2000,1.377900,0.954510,45.336800,25.648400
2500,1.106900,0.705165,54.462500,27.484400
3000,1.024000,0.634142,58.827000,27.304700


/home/codespace/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


decoded_preds[0] [BOS] Il [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
decoded_labels[0] ['Le grand Meaulnes']
decoded_preds[0] [BOS] Le grand Meaulnes [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
decoded_labels[0] ['Le grand Meaulnes']
decoded_preds[0] [BOS] Le grand Meaulnes [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [P

TrainOutput(global_step=3000, training_loss=2.67021316464742, metrics={'train_runtime': 1575.6544, 'train_samples_per_second': 40.618, 'train_steps_per_second': 1.904, 'total_flos': 79677619255296.0, 'train_loss': 2.67021316464742, 'epoch': 500.0})

## Протестируем модель

In [32]:
# Сохраняем модель и токенайзер
fast_tokenizer.save_pretrained("./transformer_attention_tokenizer")
transformer_attention_model.to('cpu').save_pretrained("./transformer_attention_model", safe_serialization=False)

In [34]:
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

# DataCollator отвечает за объединение данных в батчи -- добивает предложения до одной длинны (делает паддинг)
# преобразует numpy.array или питоновские списки в torch.Tensor

import numpy as np
import evaluate

metric = evaluate.load("sacrebleu")

loaded_tokenizer = AutoTokenizer.from_pretrained("./transformer_attention_tokenizer/")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = loaded_tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, loaded_tokenizer.pad_token_id)
    decoded_labels = loaded_tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != loaded_tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


test_transformer_config = TransformerAttentionConfig(
    pad_token_id=loaded_tokenizer.pad_token_id,
    bos_token_id=loaded_tokenizer.bos_token_id,
    eos_token_id=loaded_tokenizer.eos_token_id,
)

transformer_attention_model_loaded = Seq2SeqTransformerAttention(test_transformer_config)
# не раздедебажил, почему, но .from_pretrained не работает тут
transformer_attention_model_loaded.load_state_dict( torch.load("transformer_attention_model/pytorch_model.bin") )

training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_opus_books_model",
    eval_strategy="steps",
    eval_steps=1000,
    learning_rate=1e-4,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1000,
    predict_with_generate=True,
    logging_steps=25,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=loaded_tokenizer, return_tensors="pt")


trainer = Seq2SeqTrainer(
    model=transformer_attention_model_loaded,
    args=training_args,
    eval_dataset=books_preprocessed["train"].select(range(128)),  # валидировать будем тоже на обучающих данных (дисклаймер: это можно делать только для тестирования)
    tokenizer=fast_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


evaluate_result = trainer.evaluate(test_dataset=books_preprocessed["train"].select(range(128)))

print("evaluate_result", evaluate_result)

assert evaluate_result['eval_bleu'] >= 50

/tmp/ipykernel_2440/2433740778.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


evaluate_result {'eval_loss': 0.6341416239738464, 'eval_model_preparation_time': 0.001, 'eval_bleu': 58.827, 'eval_gen_len': 27.3047, 'eval_runtime': 12.3524, 'eval_samples_per_second': 10.362, 'eval_steps_per_second': 0.486}


Если ассерт выше прошел, можете загрузить ноутбук и обученные модельки+конфиги на гитхаб. На гитхабе из ноутбука будет выгружена ячейка с описанием модели и будет запускаться тест аналогичный ячейке выше.

Веса и конфиги надо заархивировать и на гитхаб загрузить в виде архива с сохранением названий: `rnn_tokenizer.zip`, `rnn_attention_model.zip`

Ноутбук надо загрузить с таким же названием, какое было в репозитории.


Важно! Архив с весами и конфигом модели не должен весить больше 100МБ иначе этот файлик не будет скачан в пайплайнах даже если вы его загрузите на гитхаб через git-lfs

In [35]:
!zip -r transformer_attention_model.zip transformer_attention_model
!zip -r transformer_attention_tokenizer.zip transformer_attention_tokenizer
!ls -ltrh | tail -n2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  adding: transformer_attention_model/ (stored 0%)
  adding: transformer_attention_model/config.json (deflated 38%)
  adding: transformer_attention_model/pytorch_model.bin (deflated 8%)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  adding: transformer_attention_tokenizer/ (stored 0%)
  adding: transformer_attention_tokenizer/tokenizer_config.json (deflated 72%)
  adding: transformer_attention_tokenizer/tokenizer.json (deflated 83%)
  adding: transformer_attention_tokenizer/special_tokens_map.json (deflated 42%)
-rw-rw-rw-  1 codespace root       94K Apr 28 12:25 hw_transformer_attention.ipynb
-rw-rw-rw-  1 codespace codespace 236K Apr 28 12:25 transformer_attention_tokenizer.zip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Мы зааихивировали конфиги и веса, скачайте архивы из колаба и загрузите в github репозиторий. **Не забудьте обновить ноутбук в github репозитории тоже!**


Архив с моделькой может занимтаь больше 25 мегабайт. Гитхаб не разрешает грузить большие файлы через веб-интерфейс. Можете закоммитить эти архивы через консоль или погуглить, как это сделать по-другому
https://bytesbin.com/upload-files-larger-than-25mb-to-github/
https://www.google.com/search?q=Fix+GitHub+%E2%80%98Yowza+That%E2%80%99s+a+Big+File%E2%80%99

# Вопросы!

## Почему статья называется "Attention Is All You Need"?

Механизм внимания затащил

## Почему обучение декодера RNN нельзя распараллелить, как это делается Transformer Decoder с помощью subsequent mask?

RNN (включая GRU из прошлого дз) обрабатывают последовательности пошагово, RNN не имеет механизма внимания, который позволяет "заглядывать" на любые позиции.